# Hindi voice assistant: wake word "श्रुति" + your voice only + mBERT command understanding

**Pipeline:** mic (browser, noise-suppressed) → room-noise filter → **Shruti** (speech to text) and **WavLM** (is this *your* voice?) in parallel → wake word / 30 s sleep logic → **mBERT tagger** (finds commands from the meaning of the sentence) → an **array** such as `["forward","right"]`.

**Before you run**
1. Use a **fresh runtime** with a **GPU** (T4 is fine). Do not reuse a session where torch/torchvision were reinstalled.
2. Run the cells in order. The first run downloads Shruti, WavLM and mBERT, then **trains the mBERT tagger (about 5-10 minutes on a T4)** on generated Hindi / Hinglish / English sentences.
3. In the widget: **Start listening**, stay quiet for a second (room-noise profile), click **Enrol my voice** and read the six sentences. Then say **"श्रुति"** and give commands. It sleeps after 30 s without your voice.

Emergency words (रुको, बचाओ, help, stop...) return `["stop"]` even when it is asleep or someone else says them. Set `EMERGENCY_BYPASS = False` in the settings cell to turn that off.

In [ ]:
!pip install -q -U huggingface_hub
!pip install -q -U cactus-needle transformers noisereduce soundfile
!hf download udgamlabs/Shruti shruti.cact --local-dir /content

In [ ]:
import os, sys, time, json, re, math, wave, random, unicodedata
os.environ["NEEDLE_TELEMETRY"] = "0"
import numpy as np, torch

# ================= settings (edit freely) =================
IDLE_SECONDS     = 30      # silence from YOU before it goes back to sleep
USE_DENOISE      = True    # stationary background-noise filter (fans, hum, hiss)
DENOISE_STRENGTH = 0.6     # 0..1, higher = more aggressive
NEGATION_POLICY  = "stop"  # "don't turn left" -> "stop" (safe) or "ignore"
MIN_CONF         = 0.5     # drop command spans the model is less sure of
EMERGENCY_BYPASS = True    # emergency words stop it even when asleep / from other voices
VOICEPRINT       = "/content/voiceprint.npz"
LOG_FILE         = "/content/command_log.jsonl"
NLU_MODEL        = "bert-base-multilingual-cased"   # or "distilbert-base-multilingual-cased" (about 2x faster)
NLU_DIR          = "/content/nlu_model"

dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)

# ================= ASR: Shruti (Whistle) =================
from needle import Whistle
asr = Whistle("/content/shruti.cact")

def asr_text(x):
    path = "/content/utt.wav"
    pcm = (np.clip(x, -1, 1) * 32767).astype(np.int16)
    with wave.open(path, "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(16000)
        w.writeframes(pcm.tobytes())
    return asr.transcribe(path)["text"].strip()

# ================= speaker verification: WavLM x-vector =================
from transformers import AutoFeatureExtractor, WavLMForXVector
sv_fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus-sv")
sv = WavLMForXVector.from_pretrained("microsoft/wavlm-base-plus-sv").to(dev).eval()

@torch.inference_mode()
def spk_embed(x):
    if len(x) < 19200:                                   # very short commands: repeat to reach 1.2 s
        x = np.tile(x, math.ceil(19200 / max(1, len(x))))
    inp = sv_fe(x, sampling_rate=16000, return_tensors="pt")
    e = sv(**{k: v.to(dev) for k, v in inp.items()}).embeddings
    return torch.nn.functional.normalize(e, dim=-1)[0].float().cpu().numpy()

print("ASR + speaker model ready")

### Language data, labels and emergency words
Generates labelled training sentences (commands, negations, contrasts, several commands in one sentence, chatter that only *contains* direction words) and defines the wake-word matcher.

In [ ]:
import random, re, unicodedata

def norm(s):
    s = unicodedata.normalize("NFC", s).lower()
    s = re.sub(r"[।॥.,?!;:\"'()\-–—…]", " ", s)
    return " ".join(s.split())

MOVE = ["forward", "backward", "left", "right"]
CMDS = MOVE + ["start", "stop"]
LABELS = ["O"] + [f"{b}-{pre}{c}" for c in CMDS for pre in ("", "neg_") if not (pre and c == "stop") for b in "BI"]
L2I = {l: i for i, l in enumerate(LABELS)}
I2L = {i: l for l, i in L2I.items()}

# ---------------- affirmative command phrases ----------------
AFF = {
 "forward": ["आगे चलो","आगे जाओ","आगे बढ़ो","आगे बढ़ें","आगे चलिए","आगे बढ़िए","सीधा चलो","सीधे चलो","सीधे जाओ",
             "आगे की तरफ जाओ","आगे की ओर चलो","फॉरवर्ड चलो","फॉरवर्ड","aage chalo","aage jao","aage badho","seedha chalo",
             "forward chalo","go forward","move forward","go straight","move ahead","forward","keep going forward","drive forward"],
 "backward": ["पीछे चलो","पीछे जाओ","पीछे हटो","पीछे आओ","पीछे चलिए","पीछे की तरफ जाओ","पीछे की ओर हटो","बैक करो","रिवर्स करो",
              "रिवर्स","वापस पीछे जाओ","peeche chalo","peeche jao","peeche hato","back karo","go back","move back","go backward",
              "reverse","back up","move backward"],
 "left": ["बाएं मुड़ो","बायें मुड़ो","बाएँ मुड़ो","बाईं तरफ जाओ","बाएं चलो","बाएं घूमो","बाएं हाथ की तरफ मुड़ो","बाईं ओर मुड़ो",
          "लेफ्ट मुड़ो","लेफ्ट लो","लेफ्ट","बाएं जाओ","बाएं घूम जाओ","बाएँ तरफ मुड़ जाओ","baye mudo","left mudo","left chalo",
          "baayein mudo","turn left","go left","left turn","move left","take a left","bear left"],
 "start": ["शुरू करो","चालू करो","स्टार्ट करो","चलना शुरू करो","चल पड़ो","चलना शुरू कीजिए","स्टार्ट","चालू","shuru karo",
           "chalu karo","start karo","start","start moving","begin moving"],
 "stop": ["रुको","रुक जाओ","रुक जा","रुकिए","रुक जाइए","ठहरो","रोको","रोक दो","थम जाओ","स्टॉप","स्टॉप करो","अब बस","बस करो",
          "रुको रुको","बंद करो","stop","stop moving","halt","wait","hold on","stop it","ruko","ruk jao","बचाओ","बचाव","मदद करो",
          "मदद","help","help me","खतरा","खतरा है","संभलो","सावधान","careful","नहीं नहीं","अरे रुको","मुझे बचाओ","बचाव मुझे",
          "बचाओ मुझे","ओह नो"],
}
def mirror(s):
    for a, b in [("बाएं","दाएं"),("बायें","दायें"),("बाएँ","दाएँ"),("बाईं","दाईं"),("लेफ्ट","राइट"),("baayein","daayein"),
                 ("baye","daaye"),("left","right")]:
        s = s.replace(a, b)
    return s
AFF["right"] = [mirror(p) for p in AFF["left"]]

DIRW  = {"forward":["आगे","सीधा","सीधे"], "backward":["पीछे"], "left":["बाएं","बायें","बाएँ","बाईं"], "right":["दाएं","दायें","दाएँ","दाईं","दाहिने"]}
DIRL  = {"forward":["aage","seedha"], "backward":["peeche"], "left":["baye","left"], "right":["daaye","right"]}
_t = ["मुड़ो","जाओ","घूमो","चलो"]
IMP   = {"forward":["जाओ","चलो","बढ़ो"], "backward":["जाओ","हटो","चलो","आओ"], "left":_t, "right":_t}
INF   = {"forward":["जाना","चलना","बढ़ना"], "backward":["जाना","हटना","चलना"], "left":["मुड़ना","जाना","घूमना"], "right":["मुड़ना","जाना","घूमना"]}
IMPL  = {"forward":["jao","chalo","badho"], "backward":["jao","hato","chalo"], "left":["mudo","jao","chalo"], "right":["mudo","jao","chalo"]}
EN_VP = {"forward":[("go","forward"),("move","forward"),("go","straight"),("drive","forward"),("move","ahead")],
         "backward":[("go","back"),("move","back"),("go","backward"),("move","backward"),("back","up")],
         "left":[("turn","left"),("go","left"),("move","left")], "right":[("turn","right"),("go","right"),("move","right")]}
EN_NEG = [["don","t"],["dont"],["do","not"],["never"]]

FILLER = ["अरे वाह","क्या बात है","बहुत बढ़िया","बहुत अच्छा","ठीक है","अच्छा सुनो","सुनो भाई","देखो","हाँ तो","वाह","हम्म",
          "तो फिर","अब ज़रा","एक काम करो","जरा सुनना","अच्छा तो","ऐसे ही","थैंक यू","okay","सब ठीक चल रहा है","कितना मज़ा आ रहा है",
          "देखो कितना बढ़िया काम हो रहा है","सब कुछ ठीक है","मज़ा आ गया","बहुत खूब","शाबाश","क्या बात है भाई","आज मौसम बहुत सुहावना है",
          "मुझे बहुत अच्छा लग रहा है","देखते हैं क्या होता है","कमाल है","well","so","listen","hmm","nice","good job"]
VOC    = ["श्रुति","shruti","अरे श्रुति","सुनो श्रुति","hey shruti","ok shruti","श्रुति सुनो"]
POLITE = ["कृपया","ज़रा","थोड़ा","जरा","please","प्लीज","अब","अभी","जल्दी","ज़रा सा","now"]
SUFFIX = ["प्लीज","please","जल्दी","अभी","ठीक है","समझे","okay","ना","जी"]
SUBJ   = ["तुम","आप","तुम्हें","अब तुम","यार"]
JOIN   = ["फिर","और फिर","उसके बाद","और","फिर उसके बाद","तब","phir","aur phir","uske baad","aur","then","and then","and","after that","and after that"]
CHATTER = [
 "वह बाएं हाथ से लिखता है","मैं दाएं बाएं देख रहा था","कल आगे की बात करेंगे","पीछे वाली सीट खाली है","उसने सीधा जवाब दिया","आगे का रास्ता खराब है",
 "बाएं तरफ एक दुकान है","दाएं तरफ अस्पताल है","मेरा दाहिना हाथ थक गया है","आगे चलकर यह बात समझ आएगी","पीछे मुड़कर मत देखो","आगे वाला व्यक्ति कौन है",
 "वह पीछे बैठा है","सीधा सादा आदमी है","मैं बाएं हाथ का खिलाड़ी हूँ","यह रास्ता सीधे घर जाता है","आज शुरू से ही मौसम अच्छा है","फिल्म शुरू हो गई",
 "मैं कल काम शुरू करूंगा","he writes with his left hand","the shop is on the left side","you are absolutely right","i left my keys at home",
 "right now i am busy","go to the market tomorrow","we went forward with the plan","look forward to meeting you","right on time","he took a left yesterday",
 "the movie will start at six","the start of the week was busy","the car is behind us","let me know when you are ready","आज मौसम कैसा है",
 "तुम्हारा नाम क्या है","what time is it","गाना सुनाओ","कितने बजे हैं","मुझे भूख लगी है","नमस्ते","पानी लाओ","कैसे हो","दरवाजा खोलो","आज क्या दिन है",
 "how are you","tell me a joke","what is the weather today","तुम्हारा काम बहुत अच्छा है","मुझे घर जाना है"]
REPORT = ["उसने कहा था कि","वह बोल रहा था","मैंने कल कहा था","अगर मैं कहूँ तो","जब मैं बोलूं तब","he said","if i say","she told me to","yesterday he said","माँ ने कहा था"]

def span(words, label, neg=False):
    name = ("neg_" if neg else "") + label
    return [(w, ("B-" if i == 0 else "I-") + name) for i, w in enumerate(words)]
def plain(words): return [(w, "O") for w in words]

class Gen:
    def __init__(self, seed=7): self.R = random.Random(seed)
    def aff_unit(self, c): return span(self.R.choice(AFF[c]).split(), c)
    def neg_unit(self, c):
        R = self.R
        if c == "start":
            w, k = R.choice([(["शुरू","मत","करो"],3),(["चालू","मत","करो"],3),(["स्टार्ट","मत","करो"],3),(["शुरू","नहीं","करना","है"],3),
                             (["don","t","start"],3),(["do","not","start"],3),(["shuru","mat","karo"],3)])
            return span(w[:k], c, True) + plain(w[k:])
        form = R.choice(["hi1","hi2","hi3","lat1","lat2","en","en"])
        d = R.choice(DIRW[c])
        if form == "hi1": w, k = [d, "मत", R.choice(IMP[c])], 3
        elif form == "hi2":
            w = [d, "नहीं", R.choice(INF[c])] + R.choice([[], ["है"], ["है"]]); k = 3
        elif form == "hi3": w, k = [d, "की", "तरफ", "मत", R.choice(IMP[c])], 5
        elif form == "lat1": w, k = [R.choice(DIRL[c]), "mat", R.choice(IMPL[c])], 3
        elif form == "lat2": w = [R.choice(DIRL[c]), "nahi", "jana"] + R.choice([[], ["hai"]]); k = 3
        else:
            n = R.choice(EN_NEG); v = R.choice(EN_VP[c]); w = n + list(v); k = len(w)
        return span(w[:k], c, True) + plain(w[k:])
    def contrast_unit(self, a, b):
        R = self.R
        if R.random() < 0.6:
            return span([R.choice(DIRW[a]), "नहीं"], a, True) + span([R.choice(DIRW[b]), R.choice(IMP[b])], b)
        v = R.choice(EN_VP[b])
        return span(list(v), b) + span(["not", {"forward":"forward","backward":"back","left":"left","right":"right"}[a]], a, True)
    def positive(self):
        R = self.R; r = R.random()
        if r < 0.12: units = [self.neg_unit(R.choice(MOVE + ["start"]))]
        elif r < 0.18:
            a, b = R.sample(MOVE, 2); units = [self.contrast_unit(a, b)]
        else:
            units = []
            for _ in range(R.choice([1, 1, 1, 1, 2, 2, 2, 3])):
                if R.random() < 0.10: units.append(self.neg_unit(R.choice(MOVE + ["start"])))
                else: units.append(self.aff_unit(R.choices(CMDS, weights=[3, 2, 3, 3, 1, 2])[0]))
        out = []
        if R.random() < 0.25: out += plain(R.choice(VOC).split())
        if R.random() < 0.35: out += plain(" ".join(R.sample(FILLER, R.randint(1, 3))).split())
        if R.random() < 0.10: out += plain(R.choice(SUBJ).split())
        for i, u in enumerate(units):
            if i and R.random() < 0.85: out += plain(R.choice(JOIN).split())
            if R.random() < 0.25: out += plain(R.choice(POLITE).split())
            out += u
        if R.random() < 0.25: out += plain(R.choice(SUFFIX).split())
        words = [w for w, _ in out]; tags = [t for _, t in out]
        if R.random() < 0.04: words[-1] += R.choice(["त", "ं", "ा"])         # stray ASR character
        return words, tags
    def negative(self):
        R = self.R; r = R.random()
        if r < 0.30: s = " ".join(R.sample(FILLER, R.randint(2, 6)))
        elif r < 0.75:
            s = R.choice(CHATTER)
            if R.random() < 0.4: s = " ".join(R.sample(FILLER, R.randint(1, 2))) + " " + s
            if R.random() < 0.3: s = s + " " + R.choice(FILLER)
        else:
            s = R.choice(REPORT) + " " + R.choice(AFF[R.choice(MOVE + ["start"])])
        w = s.split(); return w, ["O"] * len(w)
    def make(self, n, p_pos=0.62):
        return [self.positive() if self.R.random() < p_pos else self.negative() for _ in range(n)]

# ---------------- teach-by-markup:  [forward आगे जाओ] फिर [!left बाएं नहीं मुड़ना] ----------------
def parse_markup(s):
    words, tags = [], []
    for part in re.split(r"(\[[^\]]+\])", s):
        part = part.strip()
        if not part: continue
        m = re.match(r"\[(!?)(\w+)\s+(.+)\]", part)
        if m:
            neg, lab, txt = m.group(1) == "!", m.group(2), norm(m.group(3)).split()
            for w, t in span(txt, lab, neg): words.append(w); tags.append(t)
        else:
            for w in norm(part).split(): words.append(w); tags.append("O")
    return words, tags

# ---------------- decode ----------------
def decode_spans(words, labels, confs):
    spans, cur = [], None
    for w, lab, cf in zip(words, labels, confs):
        if lab == "O": cur = None; continue
        bi, name = lab.split("-", 1)
        if bi == "B" or cur is None or cur["name"] != name:
            neg = name.startswith("neg_")
            cur = {"name": name, "cmd": name[4:] if neg else name, "neg": neg, "words": [w], "confs": [cf]}; spans.append(cur)
        else:
            cur["words"].append(w); cur["confs"].append(cf)
    for s in spans: s["conf"] = min(s["confs"])
    return spans

def spans_to_commands(spans, min_conf=0.5, policy="stop"):
    good = [s for s in spans if s["conf"] >= min_conf]
    aff = [s["cmd"] for s in good if not s["neg"]]
    if aff: return aff
    if policy == "stop" and any(s["neg"] for s in good): return ["stop"]
    return []


HAND_TESTS = [
 ("तुम्हें बाएँ नहीं मुड़ना है", ["stop"]),
 ("तो मैं आगे नहीं चलना है", ["stop"]),
 ("don t turn right", ["stop"]),
 ("बचाव मुझे", ["stop"]),
 ("बचाओ", ["stop"]),
 ("आगे जाओ फिर दाएं मुड़ो", ["forward", "right"]),
 ("आगे जाओ फिर दाएं मुड़ो फिर रुको", ["forward", "right", "stop"]),
 ("बाएं नहीं दाएं मुड़ो", ["right"]),
 ("अरे वह क्या बात है कितना अच्छा चल रहा है ऐसे ही काम करो आगे बढ़ोत", ["forward"]),
 ("वह बाएं हाथ से लिखता है", []),
 ("कल आगे की बात करेंगे", []),
 ("पीछे वाली सीट पर कौन बैठा है", []),
 ("go forward and then turn left", ["forward", "left"]),
 ("pehle aage chalo phir peeche hato", ["forward", "backward"]),
 ("श्रुति दाएं मुड़ो", ["right"]),
 ("थोड़ा सा पीछे हटो", ["backward"]),
 ("ज़रा बाएं घूम जाओ प्लीज", ["left"]),
 ("उसने कहा था कि आगे चलो", []),
 ("अगर मैं कहूँ तो आगे चलना", []),
 ("आगे मत जाओ फिर दाएं मुड़ो", ["right"]),
 ("start moving and then go straight", ["start", "forward"]),
 ("बाएं मुड़ो बाएं मुड़ो", ["left", "left"]),
 ("अब रुक जाओ", ["stop"]),
 ("वाह क्या बात है", []),
 ("turn right then stop", ["right", "stop"]),
 ("चालू करो और आगे बढ़ो", ["start", "forward"]),
 ("पीछे मत हटो", ["stop"]),
 ("do not go left", ["stop"]),
 ("आगे बढ़ो आगे बढ़ो", ["forward", "forward"]),
 ("look forward to meeting you", []),
 ("he took a left yesterday", []),
 ("सुनो भाई ज़रा दाईं तरफ जाओ", ["right"]),
 ("पहले लेफ्ट लो फिर सीधे चलो", ["left", "forward"]),
 ("आज बहुत अच्छा दिन है चलो आगे चलते हैं", ["forward"]),
]


import unicodedata, difflib

def canon(t):
    return unicodedata.normalize("NFC", t).replace("ँ", "ं").replace("़", "")

# ---- emergency words: always -> stop, from anyone, awake or asleep (set EMERGENCY_BYPASS = False to disable) ----
NEG_TOKENS   = {canon(w) for w in ["नहीं","नही","मत","not","never","no","dont","cant","wont","नो","नॉट","डोंट"]}
PANIC_PREFIX = tuple(canon(w) for w in ["रुक","रूक","रोक","ठहर","थम","स्टॉप","बचा","मदद","खतर","संभल","संभाल","सावधान","हेल्प","बंद"])
PANIC_EXACT  = {"stop","help","wait","halt","danger","careful","emergency","whoa","ruko","bachao","madad"}
FILLER_NEG   = {"अरे","please","प्लीज","ओह"}

def panic(toks):
    if any(t in PANIC_EXACT or t.startswith(PANIC_PREFIX) for t in toks): return True
    if len(toks) <= 3 and "बस" in toks: return True
    neg = [i for i, t in enumerate(toks) if t in NEG_TOKENS]
    if any(b - a == 1 for a, b in zip(neg, neg[1:])): return True
    if toks and all(t in NEG_TOKENS or t in FILLER_NEG for t in toks): return True
    return False

# ---- wake word "shruti": fuzzy match on the first few words of what the ASR heard ----
WAKE_VARIANTS = ["श्रुति","श्रुती","श्रूति","शृति","श्रुथि","शुरुति","शुरूति","शरुति","सुरुति","श्रुत","shruti","shruthi","shuruti",
                 "sruti","shrooti","shurti","surti","shruthy"]
_WAKE = [canon(v) for v in WAKE_VARIANTS]
def find_wake(words, max_pos=4, min_ratio=0.8):
    for i, w in enumerate(words[:max_pos]):
        ww = canon(w)
        if any(difflib.SequenceMatcher(None, ww, v).ratio() >= min_ratio for v in _WAKE):
            return i
    return None


print(len(LABELS), 'labels;', len(HAND_TESTS), 'hand-written test sentences')

### mBERT command tagger
Every word gets a tag (which command it belongs to, and whether it is negated). Order and count of commands come from the tags, so two commands give an array of length 2. It trains once and is saved to `/content/nlu_model`.

In [ ]:
# ================= mBERT command tagger =================
from transformers import AutoTokenizer, AutoModelForTokenClassification, get_linear_schedule_with_warmup
MAX_LEN = 64

if os.path.exists(NLU_DIR):
    tok = AutoTokenizer.from_pretrained(NLU_DIR)
    nlu = AutoModelForTokenClassification.from_pretrained(NLU_DIR).to(dev).eval()
    print("loaded trained tagger from", NLU_DIR)
else:
    tok = AutoTokenizer.from_pretrained(NLU_MODEL)
    nlu = AutoModelForTokenClassification.from_pretrained(
        NLU_MODEL, num_labels=len(LABELS), id2label=I2L, label2id=L2I).to(dev)

@torch.inference_mode()
def predict_spans(text):
    words = norm(text).split()
    if not words:
        return []
    enc = tok(words, is_split_into_words=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
    wid = enc.word_ids(0)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(dev == "cuda")):
        logits = nlu(**{k: v.to(dev) for k, v in enc.items()}).logits[0]
    probs = logits.float().softmax(-1).cpu().numpy()
    labels, confs, ws, prev = [], [], [], None
    for j, w in enumerate(wid):
        if w is not None and w != prev:                  # first sub-token of each word carries the label
            k = int(probs[j].argmax())
            labels.append(I2L[k]); confs.append(float(probs[j][k])); ws.append(words[w])
        prev = w
    return decode_spans(ws, labels, confs)

def commands_from(text):
    """Returns (array_of_commands, per-command detail)."""
    spans = predict_spans(text)
    cmds = spans_to_commands(spans, MIN_CONF, NEGATION_POLICY)
    detail = [{"cmd": s["cmd"], "neg": s["neg"], "conf": round(s["conf"], 3), "words": " ".join(s["words"])} for s in spans]
    return cmds, detail

def evaluate(verbose=True):
    nlu.eval(); ok = 0
    for text, gold in HAND_TESTS:
        pred, _ = commands_from(text); ok += pred == gold
        if verbose and pred != gold:
            print("MISS", text, "| expected", gold, "| got", pred)
    print(f"{ok}/{len(HAND_TESTS)} exact-match on the hand-written sentences")
    return ok / len(HAND_TESTS)

def _collate(items):
    enc = tok([w for w, _ in items], is_split_into_words=True, truncation=True, max_length=MAX_LEN,
              padding=True, return_tensors="pt")
    labels = torch.full(enc["input_ids"].shape, -100)
    for i, (_, tags) in enumerate(items):
        prev = None
        for j, wid in enumerate(enc.word_ids(i)):
            if wid is not None and wid != prev:
                labels[i, j] = L2I[tags[wid]]
            prev = wid
    enc["labels"] = labels
    return {k: v.to(dev) for k, v in enc.items()}

def train(epochs=3, n=14000, extra=None, lr=3e-5, bs=32, seed=0):
    """extra = list of markup strings, e.g. '[forward आगे जाओ] फिर [!left बाएँ नहीं मुड़ना]' (see ADD_EXAMPLES)."""
    data = Gen(seed).make(n)
    if extra:
        data += [parse_markup(m) for m in extra] * 25
    rnd = random.Random(seed)
    steps = epochs * math.ceil(len(data) / bs)
    opt = torch.optim.AdamW(nlu.parameters(), lr=lr, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    amp = dev == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    for ep in range(epochs):
        nlu.train(); rnd.shuffle(data); t0 = time.time(); tot = 0.0
        for i in range(0, len(data), bs):
            batch = _collate(data[i:i + bs])
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
                loss = nlu(**batch).loss
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(nlu.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
            tot += float(loss)
        print(f"epoch {ep + 1}/{epochs}  loss {tot / math.ceil(len(data) / bs):.4f}  ({time.time() - t0:.0f}s)")
        evaluate(verbose=(ep == epochs - 1))
    nlu.eval()
    nlu.save_pretrained(NLU_DIR); tok.save_pretrained(NLU_DIR)

# Sentences the model got wrong in real use go here, labelled with markup, then run:  train(epochs=2, n=3000, extra=ADD_EXAMPLES)
#   [forward ...] = affirmative command words,  [!left ...] = a negated command,  everything else = ignored
ADD_EXAMPLES = [
    # "[forward आगे जाओ] फिर [right दाएं मुड़ो]",
]

if not os.path.exists(NLU_DIR):
    train(epochs=3)
else:
    evaluate()

### Assistant logic (wake word, your voice, noise filter, 30 s sleep) and browser callbacks

In [ ]:
import time, json, traceback
from base64 import b64decode
from concurrent.futures import ThreadPoolExecutor
from IPython.display import JSON
from google.colab import output
POOL = ThreadPoolExecutor(max_workers=2)


class State:
    awake = False
    last_active = 0.0
    noise = None
    enrolled = False
    centroid = None
    thr_strict = 0.84
    thr_loose = 0.72
    enroll_embs = []
S = State()

def pcm_from_b64(b64):
    return np.frombuffer(b64decode(b64), dtype=np.int16).astype(np.float32) / 32768.0

def denoise(x):
    if not USE_DENOISE or S.noise is None or len(x) < 4096:
        return x
    import noisereduce as nr
    y = nr.reduce_noise(y=x, sr=16000, y_noise=S.noise, stationary=True, prop_decrease=DENOISE_STRENGTH)
    return np.clip(y, -1, 1).astype(np.float32)

def spk_sim(x):
    e = spk_embed(x)
    return float(np.dot(e, S.centroid))

def finalize_enroll():
    E = np.stack(S.enroll_embs)
    c = E.mean(0); c = c / np.linalg.norm(c)
    g = []
    for i in range(len(E)):                       # leave-one-out: how similar is each sample to the rest?
        ci = np.delete(E, i, 0).mean(0); ci = ci / np.linalg.norm(ci); g.append(float(E[i] @ ci))
    g = np.array(g)
    strict = float(np.clip(g.mean() - 1.5 * g.std() - 0.02, 0.72, 0.88))
    loose  = float(np.clip(g.min() - 0.05, 0.62, strict - 0.05))
    S.centroid, S.thr_strict, S.thr_loose, S.enrolled = c, strict, loose, True
    np.savez(VOICEPRINT, centroid=c, strict=strict, loose=loose)
    return {"samples": len(E), "genuine_mean": float(g.mean()), "genuine_min": float(g.min()),
            "wake_threshold": strict, "awake_threshold": loose}

def load_voiceprint():
    if os.path.exists(VOICEPRINT):
        d = np.load(VOICEPRINT)
        S.centroid, S.thr_strict, S.thr_loose, S.enrolled = d["centroid"], float(d["strict"]), float(d["loose"]), True
        return True
    return False

def _py(o):
    if isinstance(o, dict): return {k: _py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_py(v) for v in o]
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.integer,)): return int(o)
    return o

def handle(b64, dur):
    t_all = time.time()
    x = denoise(pcm_from_b64(b64))
    fa = POOL.submit(asr_text, x)
    fs = POOL.submit(spk_sim, x) if S.enrolled else None
    text = fa.result(); t_asr = time.time() - t_all
    sim = fs.result() if fs else None
    now = time.time()
    if S.awake and now - S.last_active > IDLE_SECONDS:
        S.awake = False
    words = norm(text).split()
    toks = [canon(w) for w in words]
    res = {"text": text, "sim": sim, "owner": None, "cmds": [], "detail": [], "event": "ignored", "reason": ""}

    def done():
        res["awake"] = S.awake
        res["awake_for"] = max(0.0, IDLE_SECONDS - (time.time() - S.last_active)) if S.awake else 0.0
        res["ms"] = {"asr": round(t_asr * 1000), "total": round((time.time() - t_all) * 1000)}
        try:
            with open(LOG_FILE, "a", encoding="utf-8") as f:
                f.write(json.dumps(_py({k: res[k] for k in ("text", "sim", "owner", "cmds", "event", "awake")}), ensure_ascii=False) + "\n")
        except Exception:
            pass
        return _py(res)

    if not words:
        res["reason"] = "no speech recognised"; return done()

    if EMERGENCY_BYPASS and panic(toks):                                     # anyone can stop it, any time
        res.update(cmds=["stop"], event="emergency", reason="emergency word")
        if S.awake and (not S.enrolled or sim >= S.thr_loose): S.last_active = now
        return done()

    owner = True if not S.enrolled else sim >= (S.thr_loose if S.awake else S.thr_strict)
    res["owner"] = bool(owner)
    if not owner:
        res["reason"] = "not your voice"; return done()

    pos = find_wake(words)
    if not S.awake:
        if pos is None:
            res["reason"] = "asleep: say 'श्रुति' first"; return done()
        S.awake = True
        res["event"] = "wake"
    S.last_active = now
    rest = words[pos + 1:] if (pos is not None) else words
    if not rest:
        res["reason"] = "listening"; return done()

    t0 = time.time()
    cmds, detail = commands_from(" ".join(rest))
    res["ms_nlu"] = round((time.time() - t0) * 1000)
    res["cmds"], res["detail"] = cmds, detail
    if cmds:
        res["event"] = "command"
    elif res["event"] != "wake":
        res["event"] = "heard"; res["reason"] = "no command found"
    return done()


# ================= browser callbacks =================
def _noise(b64, dur):
    x = pcm_from_b64(b64)
    S.noise = x if len(x) >= 8000 else None
    return JSON({"ok": S.noise is not None})

def _utterance(b64, dur):
    try:
        return JSON(handle(b64, dur))
    except Exception:
        return JSON({"event": "error", "error": traceback.format_exc()[-600:], "cmds": [], "detail": [], "text": "", "awake": S.awake, "awake_for": 0})

def _enroll(b64, dur, total):
    try:
        x = denoise(pcm_from_b64(b64))
        if len(x) < 16000:
            return JSON({"accepted": False, "msg": "too short: say the whole sentence"})
        if float(np.sqrt(np.mean(x ** 2))) < 0.004:
            return JSON({"accepted": False, "msg": "too quiet: speak a little louder"})
        e = spk_embed(x)
        if len(S.enroll_embs) >= 2:
            c = np.mean(S.enroll_embs, 0); c = c / np.linalg.norm(c)
            if float(e @ c) < 0.55:
                return JSON({"accepted": False, "msg": "that sounded like a different voice: only you should speak during enrolment"})
        S.enroll_embs.append(e)
        n = len(S.enroll_embs)
        if n >= total:
            return JSON({"accepted": True, "n": n, "done": True, "summary": finalize_enroll()})
        return JSON({"accepted": True, "n": n, "done": False})
    except Exception:
        return JSON({"accepted": False, "msg": traceback.format_exc()[-300:]})

def _enroll_reset():
    S.enroll_embs = []; S.enrolled = False; S.awake = False
    return JSON({"ok": True})

def _sleep():
    S.awake = False
    return JSON({"ok": True})

for _name, _fn in [("va_noise", _noise), ("va_utterance", _utterance), ("va_enroll", _enroll),
                   ("va_enroll_reset", _enroll_reset), ("va_sleep", _sleep)]:
    output.register_callback("notebook." + _name, _fn)

print("voiceprint loaded from disk" if load_voiceprint() else "no voiceprint yet: use 'Enrol my voice' in the widget")

### The widget

In [ ]:
from IPython.display import HTML, display

UI = r"""
<div id="va" style="font-family:system-ui,sans-serif;max-width:740px">
  <div style="margin-bottom:8px">
    <button id="va-listen" style="font-size:15px;padding:6px 12px">▶ Start listening</button>
    <button id="va-enroll" style="font-size:15px;padding:6px 12px">🎙 Enrol my voice</button>
    <span id="va-badge" style="margin-left:10px;padding:3px 10px;border-radius:12px;background:#888;color:#fff;font-weight:600">STOPPED</span>
  </div>
  <div style="margin:6px 0;font-size:13px">
    mic level <progress id="va-lvl" max="0.2" value="0" style="width:180px;vertical-align:middle"></progress>
    &nbsp;speech threshold <input id="va-thr" type="range" min="0.005" max="0.2" step="0.005" value="0.03" style="vertical-align:middle;width:120px">
    <span id="va-thrv">0.030</span>
    &nbsp;silence ms <input id="va-sil" type="number" value="700" step="100" min="300" style="width:64px">
  </div>
  <div id="va-enrollbox" style="display:none;margin:8px 0;padding:10px;border:1px solid #888;border-radius:8px">
    <div style="font-size:13px;opacity:.8" id="va-enrollstep"></div>
    <div style="font-size:24px;margin:6px 0" id="va-prompt"></div>
    <div style="font-size:13px" id="va-enrollmsg"></div>
  </div>
  <div id="va-cmds" style="font-size:34px;font-family:monospace;margin:10px 0">[ ]</div>
  <div id="va-text" style="font-size:19px;margin-bottom:4px"></div>
  <div id="va-meta" style="font-size:13px;opacity:.75;margin-bottom:6px"></div>
  <ul id="va-log" style="font-size:13px;margin-top:10px"></ul>
</div>
<script>
(function(){
  const $ = id => document.getElementById(id);
  const PRE_MS = 300, MIN_MS = 300, MAX_MS = 12000;
  const PROMPTS = ["श्रुति", "श्रुति, आगे चलो", "बाएं मुड़ो और फिर दाएं मुड़ो", "पीछे हटो और रुको",
                   "turn left and then go forward", "आज मौसम बहुत अच्छा है"];
  let ctx, stream, proc, src, mute;
  let running = false, speaking = false, buf = [], pre = [];
  let lastVoice = 0, speechStart = 0, calib = [], calibUntil = 0, calibChunks = [];
  let mode = 'listen', enrollIdx = 0, awakeDeadline = 0, nProc = 0;

  $('va-thr').oninput = () => { $('va-thrv').textContent = parseFloat($('va-thr').value).toFixed(3); };

  function badge(){
    const b = $('va-badge');
    let t, c;
    if (!running){ t = 'STOPPED'; c = '#888'; }
    else if (mode === 'enroll'){ t = 'ENROLLING'; c = '#7b3fe4'; }
    else if (awakeDeadline > performance.now()){ t = 'AWAKE ' + Math.ceil((awakeDeadline - performance.now()) / 1000) + 's'; c = '#1a9a46'; }
    else { t = "SLEEPING: say \u201cश्रुति\u201d"; c = '#c47f00'; }
    b.textContent = t; b.style.background = c;
  }
  setInterval(() => {
    if (running && mode === 'listen' && awakeDeadline && awakeDeadline <= performance.now()){
      awakeDeadline = 0;
      google.colab.kernel.invokeFunction('notebook.va_sleep', [], {});
    }
    badge();
  }, 250);

  function to16k(chunks, sr){
    let n = 0; chunks.forEach(c => n += c.length);
    const all = new Float32Array(n); let o = 0;
    chunks.forEach(c => { all.set(c, o); o += c.length; });
    const ratio = sr / 16000, m = Math.floor(n / ratio), out = new Int16Array(m);
    for (let i = 0; i < m; i++){
      const a = Math.floor(i * ratio), b = Math.min(n, Math.floor((i + 1) * ratio));
      let s = 0; for (let j = a; j < b; j++) s += all[j];
      out[i] = Math.max(-1, Math.min(1, s / Math.max(1, b - a))) * 32767;
    }
    return out;
  }
  function b64(int16){
    const u8 = new Uint8Array(int16.buffer); let s = ''; const CH = 0x8000;
    for (let i = 0; i < u8.length; i += CH) s += String.fromCharCode.apply(null, u8.subarray(i, i + CH));
    return btoa(s);
  }
  const call = (name, args) => google.colab.kernel.invokeFunction(name, args, {}).then(r => r.data['application/json']);
  const hhmmss = () => new Date().toTimeString().slice(0, 8);

  function logLine(txt, dim){
    const li = document.createElement('li'); li.textContent = txt; if (dim) li.style.opacity = '.5';
    $('va-log').prepend(li);
    while ($('va-log').children.length > 10) $('va-log').lastChild.remove();
  }

  function show(d){
    if (d.error){ $('va-text').textContent = 'error: ' + d.error; logLine(hhmmss() + '  ERROR ' + d.error.slice(-120), false); return; }
    if (d.awake) awakeDeadline = performance.now() + d.awake_for * 1000; else awakeDeadline = 0;
    const sim = (d.sim === null || d.sim === undefined) ? '-' : d.sim.toFixed(2);
    const ms = d.ms ? (d.ms.total + ' ms (ASR ' + d.ms.asr + (d.ms_nlu !== undefined ? ', tagger ' + d.ms_nlu : '') + ')') : '';
    $('va-meta').textContent = 'event: ' + d.event + (d.reason ? ' (' + d.reason + ')' : '') + '  |  voice match ' + sim + '  |  ' + ms;
    if (d.event === 'command' || d.event === 'emergency'){
      $('va-cmds').textContent = JSON.stringify(d.cmds);
      $('va-text').textContent = d.text + (d.detail && d.detail.length
        ? '   ' + d.detail.map(x => (x.neg ? '!' : '') + x.cmd + ' ' + Math.round(x.conf * 100) + '%').join(', ') : '');
    } else if (d.event === 'wake' || d.event === 'heard'){
      $('va-cmds').textContent = '[ ]'; $('va-text').textContent = d.text;
    }
    logLine(hhmmss() + '  ' + d.event + '  ' + JSON.stringify(d.cmds) + '  | ' + d.text + '  | match ' + sim + (d.reason ? '  | ' + d.reason : ''),
            d.event === 'ignored' || d.event === 'heard');
  }

  function showPrompt(){
    $('va-enrollstep').textContent = 'Voice enrolment: sentence ' + (enrollIdx + 1) + ' of ' + PROMPTS.length + '. Read it aloud at normal volume.';
    $('va-prompt').textContent = PROMPTS[enrollIdx];
  }

  async function send(int16){
    nProc++;
    try {
      if (mode === 'enroll'){
        const d = await call('notebook.va_enroll', [b64(int16), int16.length / 16000, PROMPTS.length]);
        if (!d.accepted){ $('va-enrollmsg').textContent = d.msg; }
        else if (d.done){
          mode = 'listen'; $('va-enrollbox').style.display = 'none'; $('va-enroll').textContent = '🎙 Re-enrol my voice';
          const s = d.summary;
          $('va-meta').textContent = 'voice saved. wake threshold ' + s.wake_threshold.toFixed(2) + ', awake threshold ' + s.awake_threshold.toFixed(2) +
                                     ' (your own samples scored ' + s.genuine_min.toFixed(2) + '-' + s.genuine_mean.toFixed(2) + ')';
          logLine(hhmmss() + '  enrolment complete, now say "श्रुति" to wake it', false);
        } else { enrollIdx = d.n; $('va-enrollmsg').textContent = 'ok'; showPrompt(); }
      } else {
        show(await call('notebook.va_utterance', [b64(int16), int16.length / 16000]));
      }
    } catch (err) { $('va-text').textContent = 'error: ' + err; }
    nProc--;
  }

  function finish(){
    speaking = false;
    const voicedMs = lastVoice - speechStart, chunks = buf; buf = [];
    if (voicedMs < MIN_MS) return;
    send(to16k(chunks, ctx.sampleRate));
  }

  function onAudio(e){
    if (!running) return;
    const x = new Float32Array(e.inputBuffer.getChannelData(0));
    let s = 0; for (let i = 0; i < x.length; i++) s += x[i] * x[i];
    const rms = Math.sqrt(s / x.length), now = performance.now();
    $('va-lvl').value = Math.min(rms, 0.2);
    if (calibUntil){
      if (now < calibUntil){ calib.push(rms); calibChunks.push(x); return; }
      const m = calib.reduce((a, b) => a + b, 0) / Math.max(1, calib.length);
      const thr = Math.min(0.1, Math.max(0.01, m * 3));
      $('va-thr').value = thr; $('va-thr').oninput();
      call('notebook.va_noise', [b64(to16k(calibChunks, ctx.sampleRate)), calibChunks.length * 4096 / ctx.sampleRate]);
      calibUntil = 0; calib = []; calibChunks = [];
      logLine(hhmmss() + '  room noise profile captured', true);
    }
    const thr = parseFloat($('va-thr').value), sil = parseInt($('va-sil').value);
    const chunkMs = x.length / ctx.sampleRate * 1000;
    if (rms > thr){
      if (!speaking){ speaking = true; speechStart = now; buf = pre.slice(); pre = []; }
      lastVoice = now; buf.push(x);
    } else if (speaking){
      buf.push(x);
      if (now - lastVoice > sil) finish();
    } else {
      pre.push(x);
      while (pre.length * chunkMs > PRE_MS) pre.shift();
    }
    if (speaking && now - speechStart > MAX_MS) finish();
  }

  async function start(){
    stream = await navigator.mediaDevices.getUserMedia({audio: {channelCount: 1, echoCancellation: true, noiseSuppression: true, autoGainControl: false}});
    ctx = new (window.AudioContext || window.webkitAudioContext)();
    src = ctx.createMediaStreamSource(stream);
    proc = ctx.createScriptProcessor(4096, 1, 1);
    mute = ctx.createGain(); mute.gain.value = 0;
    src.connect(proc); proc.connect(mute); mute.connect(ctx.destination);
    calib = []; calibChunks = []; calibUntil = performance.now() + 1200;
    logLine(hhmmss() + '  calibrating room noise: stay quiet for a second...', true);
    proc.onaudioprocess = onAudio;
    running = true; $('va-listen').textContent = '■ Stop';
  }
  function stop(){
    running = false; speaking = false; buf = []; pre = []; awakeDeadline = 0;
    try { proc.disconnect(); src.disconnect(); mute.disconnect(); } catch (e) {}
    stream.getTracks().forEach(t => t.stop()); ctx.close();
    $('va-listen').textContent = '▶ Start listening';
  }
  $('va-listen').onclick = () => { running ? stop() : start().catch(e => logLine('mic error: ' + e, false)); };
  $('va-enroll').onclick = async () => {
    try {
      if (!running) await start();
      await call('notebook.va_enroll_reset', []);
      mode = 'enroll'; enrollIdx = 0; awakeDeadline = 0;
      $('va-enrollbox').style.display = 'block'; $('va-enrollmsg').textContent = ''; showPrompt();
    } catch (e) { logLine('mic error: ' + e, false); }
  };
})();
</script>
"""
display(HTML(UI))

### When it gets a sentence wrong
Every utterance is logged to `/content/command_log.jsonl`. Put the mistakes in `ADD_EXAMPLES` (cell with the tagger) using markup, for example
`"[forward आगे जाओ] फिर [right दाएं मुड़ो]"` (use `[!left ...]` for a negated command), then run
`train(epochs=2, n=3000, extra=ADD_EXAMPLES)`. Copy `/content/nlu_model` and `/content/voiceprint.npz` to Google Drive to keep them between sessions.